Set up the documents that our small RAG app will answer questions about.It saves everything to disk, so Notebook 02 (the actual app) can use it.

1. Set up the embedding model and the database (same as Week 7).
2. Add a small hardcoded list of documents (easy way to test).
3. Learn how to load documents from `.txt` files in a folder (the "upload" step).

In [ ]:
%pip install chromadb sentence-transformers

In [1]:
import chromadb
from sentence_transformers import SentenceTransformer

embedding_model = SentenceTransformer("all-MiniLM-L6-v2")

db_client = chromadb.PersistentClient(path="./week8_rag_db")

collection = db_client.get_or_create_collection(name="week8_documents")

print("Database is ready. Current number of chunks stored:", collection.count())

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Database is ready. Current number of chunks stored: 6


**Step 1**: Add documents from a simple Python list


In [2]:
def add_documents_to_db(documents, id_prefix="doc"):
    """
    Takes a list of text strings (documents) and stores them in the database.

    documents  -> a list of strings, e.g. ["text 1", "text 2"]
    id_prefix  -> a short label used to build unique IDs, e.g. "doc_0", "doc_1"
    """

    if not documents:
        print("No documents were given, so nothing was added.")
        return

    # Builds a unique ID for every document, e.g. doc_0, doc_1, doc_2 ...
    ids = [f"{id_prefix}_{i}" for i in range(len(documents))]


    embeddings = embedding_model.encode(documents, convert_to_numpy=True)

    # upsert = "update if it exists, insert if it does not".
    collection.upsert(
        ids=ids,
        documents=documents,
        embeddings=embeddings.tolist()
    )

    print(f"Added {len(documents)} chunks. Total chunks in database now: {collection.count()}")


sample_documents = [
    "Streamlit is a Python library that turns a normal script into a simple web app.",
    "A RAG application first retrieves relevant text, then sends it to an LLM to answer.",
    "Error handling means writing code that reacts calmly to problems instead of crashing.",
    "A try/except block lets your program catch an error and keep running.",
    "ChromaDB is a vector database that stores embeddings and lets you search by meaning.",
    "A README file explains what a project does and how someone else can run it."
]

add_documents_to_db(sample_documents, id_prefix="sample")

Added 6 chunks. Total chunks in database now: 6


**Step 2:** Load documents from your own `.txt` files.
Put a few `.txt` files inside it, and this function will read them all in.

In [3]:
import os

def load_documents_from_folder(folder_path):
    """
    Reads every .txt file in a folder, splits each file into paragraph-sized
    chunks, and stores those chunks in the database.
    """
    # os.path.exists checks if the folder is really there, if not it creates.
    if not os.path.exists(folder_path):
        print(f"Folder '{folder_path}' was not found. Create it and add .txt files first.")
        return

    txt_files = [f for f in os.listdir(folder_path) if f.endswith(".txt")]

    if not txt_files:
        print(f"No .txt files were found inside '{folder_path}'.")
        return

    all_chunks = []

    for filename in txt_files:
        file_path = os.path.join(folder_path, filename)

        # try/except: if one file has a problem,
        # we skip it and keep going instead of stopping the whole notebook.
        try:
            with open(file_path, "r", encoding="utf-8") as file:
                text = file.read()
        except Exception as error:
            print(f"Could not read {filename}, skipping it. Reason: {error}")
            continue

        # Split the file into paragraphs using blank lines.
        # .strip() removes extra spaces, and we ignore empty paragraphs.
        paragraphs = [p.strip() for p in text.split("\n\n") if p.strip()]
        all_chunks.extend(paragraphs)

    add_documents_to_db(all_chunks, id_prefix="file")


# Example usage (uncomment the line below once you have created the folder):
# load_documents_from_folder("my_documents")

Documents are now saved on disk inside the `week8_rag_db` folder.Now moving on to build the actual question-answering app.